# SACDpy batch multicolor single-z reconstruction

Ready-to-run pipeline for ONI acquisitions with one time movie at one z-plane per FOV. Each movie may contain fewer acquisition steps than the number of lasers listed in its metadata. The settings cell maps acquisition-step order to an output channel name, SACD wavelength, active metadata laser, and camera half. For every channel, the pipeline writes exactly one float32 order-root SACD image and one native-pixel uint16 time-MIP (maximum over the selected raw frames).

## 1. Setup

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import tifffile

repo = Path.cwd()
src = repo / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from sacdpy.multicolor_single_z import (
    build_batch_plan, preflight_summary, run_batch, validate_fov_outputs,
)
print(f'Working folder: {repo}')

## 2. Settings

Edit this cell for each homogeneous acquisition folder. `channels` must follow the acquisition-step order in `laserProgram.steps`, not sorted wavelength order. `name` controls the output filename, `wavelength_nm` controls the SACD PSF, `metadata_laser_nm` validates which laser was active, and `camera_half` selects the left or right half of the ONI frame. Frame boundaries come from each step's `nRepeats` metadata.

In [ ]:
# Default example: far-red signal on the right camera half.
raw_root = Path('/Volumes/gmgaoSSD/DEFAULT_USER/20260831_ONI-gmgao-IFveri/405-488-647')
output_root = raw_root / 'SACDpy_results'
channels = [
    {'name': 'AF647', 'wavelength_nm': 647.0, 'metadata_laser_nm': 640.0, 'camera_half': 'right'},
    {'name': 'AF488', 'wavelength_nm': 488.0, 'metadata_laser_nm': 488.0, 'camera_half': 'left'},
    {'name': 'Hoechst', 'wavelength_nm': 405.0, 'metadata_laser_nm': 405.0, 'camera_half': 'left'},
]

# For the reorganized 405-488-561 example, instead use:
# raw_root = Path('/Volumes/gmgaoSSD/DEFAULT_USER/20260831_ONI-gmgao-IFveri/405-488-561')
# output_root = raw_root / 'SACDpy_results'
# channels = [
#     {'name': 'AF594', 'wavelength_nm': 561.0, 'metadata_laser_nm': 561.0, 'camera_half': 'left'},
#     {'name': 'AF488', 'wavelength_nm': 488.0, 'metadata_laser_nm': 488.0, 'camera_half': 'left'},
#     {'name': 'Hoechst', 'wavelength_nm': 405.0, 'metadata_laser_nm': 405.0, 'camera_half': 'left'},
# ]

config = {
    'version': 1,
    'raw_root': str(raw_root),
    'output_root': str(output_root),
    'initial_seconds_per_reconstruction': 10.0,
    'processing': {
        'position_folder': 'pos_0',
        'glob_pattern': '*.tif',
        'pixel_nm': None,
        'fallback_pixel_nm': 117.0,
        'na': None,
        'fallback_na': 1.45,
        'mag': 2,
        'iter1': 7,
        'iter2': 8,
        'ac_order': 2,
        'intensity_transform': 'order_root',
        'subfactor': 0.8,
        'ifbackground': False,
        'backgroundfactor': 2.0,
        'ifregistration': False,
        'ifsparsedecon': False,
        'fidelity': 100.0,
        'tcontinuity': 0.1,
        'sparsity': 1.0,
        'sparse_iterations': 100,
        'channels': channels,
    },
    'selected_fov_folders': [],
    'exclude_fovs': {},
}
print(json.dumps(config, indent=2))

## 3. Read-only discovery and preflight

Run this cell before processing. It requires one matching `t0_posZ*.tif` per FOV and validates filenames, TIFF dimensions, frame totals, active laser identity for every acquisition step, camera-split width, pixel size, NA, and unique folder-derived output paths. It does not create or modify files.

In [ ]:
plan = build_batch_plan(config)
print(json.dumps(preflight_summary(plan), indent=2))
print('\nAccepted FOVs:')
for fov in plan.fovs:
    ranges = [
        f'{channel.name}: frames {start + 1}-{end}, active laser {laser:g} nm, '
        f'{channel.camera_half}, PSF {channel.wavelength_nm:g} nm'
        for channel, (start, end), laser in zip(
            fov.channels, fov.movie.frame_ranges, fov.movie.active_lasers_nm, strict=True
        )
    ]
    print(f'+ {fov.relative_fov}: t{fov.movie.time_index}, z{fov.movie.z_index}, input={fov.movie.shape}')
    print('  ' + ' | '.join(ranges))
    for output in fov.outputs:
        print(f'  -> {output.sacd.name}; {output.mip.name}')
print('\nExcluded FOVs:')
for item in plan.exclusions:
    print(f"- {item['relative_fov']}: {item['reason']}")

## 4. Run full resumable batch

This is the only processing cell. It reconstructs every configured channel at the single z-plane and writes the SACD/time-MIP pair into `SACDpy_results`. Existing complete pairs are validated before being resumed; partial or provenance-incompatible output sets are rejected. `_processing/manifest.json`, `run_status.json`, and `run.log` are updated after each FOV.

In [ ]:
results = run_batch(config)
print(f'Run returned {len(results)} result record(s).')

## 5. Validate and preview the first completed FOV

In [ ]:
preview_result = next(
    item for item in results
    if item['status'] in {'written', 'skipped_existing', 'resumed_manifest'}
)
preview_fov = next(fov for fov in plan.fovs if fov.relative_fov == preview_result['relative_fov'])
validate_fov_outputs(
    preview_fov,
    intensity_transform=config['processing']['intensity_transform'],
    mag=config['processing']['mag'],
)
cmaps = {'Hoechst': 'Blues', 'AF488': 'Greens', 'AF594': 'Oranges', 'AF647': 'Reds'}
fig, axes = plt.subplots(
    len(preview_fov.channels), 2,
    figsize=(8, 4 * len(preview_fov.channels)), squeeze=False,
)
for channel_index, channel in enumerate(preview_fov.channels):
    output = preview_fov.output_for(channel.name)
    time_mip = tifffile.imread(output.mip)
    sacd = tifffile.imread(output.sacd)
    cmap = cmaps.get(channel.name, 'gray')
    axes[channel_index, 0].imshow(time_mip, cmap=cmap)
    axes[channel_index, 0].set_title(f'{channel.name} raw time-MIP')
    axes[channel_index, 1].imshow(sacd, cmap=cmap)
    axes[channel_index, 1].set_title(f'{channel.name} order-root SACD')
    for axis in axes[channel_index]:
        axis.axis('off')
plt.tight_layout();

## 6. Final summary

In [ ]:
manifest_path = Path(config['output_root']) / '_processing' / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
statuses = {}
for item in manifest.get('fovs', []):
    statuses[item['status']] = statuses.get(item['status'], 0) + 1
print('FOV statuses:', statuses)
print('Output folder:', config['output_root'])